# Elevator Code Memory Experiment
In this demonstration we will be performing a Z-type memory experiment on a [7,4,3] elevator code using Cudaq GPU accelerated decoders. We are going to use the RelayBP decoder to decode 1 round of the outer code for multiple different $d_Z$ distances and builde a model for the X logical error rate of this code family. 

In [ ]:
from Circuit_builder import noisy_circ
import numpy as np
import cudaq_qec as qec
from stimbposd import *
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from scipy.stats import linregress
import pandas as pd
from lmfit import Model

## Memory Circuit

First, lets produce our circuit. We have created a special file for this and you just need to call `noisy_circ(dz, outer_code_rounds, X_error_probability)`.

Here is a dz=3, elevator code with a memory circuit for 1 outer code round at an X error porbability of $10^{-5}$. We will leave out the errors so that the circuit is easier to read.

In [ ]:
circuit = noisy_circ(d=3, rs=1, p=1e-5).without_noise()


html_content = f"""
<div style="overflow-x: auto; overflow-y: hidden;">
    <div style="white-space: nowrap;width: {100}%;">{circuit.diagram('timeline-svg')}</div>
</div>
"""
display(HTML(html_content))

## Detector Error Model

Now we need to build out a detector error model.

This is a graphical representation of which errors trigger which detectors on the circuit.

- An error is represented by an edge, and a detector is represented by a node.
- If a detector is triggered by a certain error, the edge will be connected to that node.
- Since this is a non-matchable code, edges can be connected to more than two nodes.

<span style="display:inline-block;width:12px;height:12px;background:black;"></span> Black edges represent matchable errors.

<span style="display:inline-block;width:12px;height:12px;background:blue;"></span> Blue edges represent non-matchable errors.

<span style="display:inline-block;width:12px;height:12px;background:red;"></span> Red edges represent edges that flip logical observables.



In [ ]:
circuit=noisy_circ(3,1,1e-5)
dem=circuit.detector_error_model(approximate_disjoint_errors=True)
dem.diagram("matchgraph-svg")

## RelayBP

Next, we need to set up our decoder. We will be using the RelayBP decoder from Cudaq-qec. This is a very powerful ensemble Belief Propagation decoder developed recently by IBM. It is also of particular importance as it may be well suited to the latency requirements of real-time decoding. The only downside is it has some of parameters that need to be set up:

- gamma0 - The initial value of the 'memory parameter' that is used in first BP leg.

- gamma_dist - The range of values that the 'memory parameter' can be in for the disordered legs.

- pre_iter - The number of iterations of Belief Propagation for the initial relay leg.

- num_sets - The number of relay legs launched after initial BP.

- stopping_criterion - The condition on which the decoder is told to hault.

In [ ]:
def nvidia_stim_decoder(dem):

    mats = detector_error_model_to_check_matrices(dem) #Creating standard H and L matricies from Detector Error Model

    H_dense = np.asarray(mats.check_matrix.toarray(), dtype=np.uint8, order='C')
    L_dense = np.asarray(mats.observables_matrix.toarray(), dtype=np.uint8, order='C')

    priors=[] #Priors are the probabilities of each error channel
    for inst in dem:
        if inst.type == "error":
            priors.append(inst.args_copy()[0])  
    priors = np.asarray(priors, dtype=np.float64)

    decoder = qec.get_decoder("nv-qldpc-decoder",  #Here we are creating the RelayBP decoder using our DEM, priors and chosen parameters
                              H_dense,
                              error_rate_vec=priors,
                              use_sparsity=True,
                              bp_method=3,
                              gamma0=0.6,
                              gamma_dist=[-0.2,0.2],
                              composition = 1,
                              srelay_config={
                                    "pre_iter": 10,
                                    "num_sets": 4,
                                    "stopping_criterion": "All",
                                },
                              bp_batch_size=1000)

    return decoder, L_dense


We will now run the decoding simulations. We will decode Z-type memory experiments with $d_Z=9$ for 1 outer code round at 5 different physical error rates $10^{-6}<p<10^{-5}$.

In [ ]:

num_shots = 1000000
ps = np.logspace(-6,-5,5)

err_rates=[]
for p in ps:
        circuit=noisy_circ(9,1,p)

        dem = circuit.detector_error_model()

        decoder, L = nvidia_stim_decoder(dem)

        mats = detector_error_model_to_check_matrices(dem)
        H = mats.check_matrix
        L = mats.observables_matrix

        sampler = circuit.compile_detector_sampler()
        shots, observables = sampler.sample(num_shots, separate_observables=True)

        predicted_observables=[]
        errors=decoder.decode_batch(shots) 

        for i in range(num_shots):
            result = errors[i].result
            data_prediction = np.array(result, dtype=np.uint8)
            predicted_observables.append((L @ data_prediction) % 2)
        num_mistakes = np.sum(np.any(predicted_observables != observables, axis=1))

        err_rates.append(num_mistakes/num_shots)

log_ps = np.log10(ps)
log_errs = np.log10(err_rates)

slope, intercept, r_value, p_value, std_err = linregress(log_ps, log_errs)

plt.scatter(ps, err_rates,  marker='o')

plt.loglog()

plt.plot(ps, 10**(intercept + slope * log_ps), label="RelayBP", linestyle='--')

plt.xlabel('Physical X Error Rate')
plt.ylabel('Logical X Error Rate')
plt.legend()

plt.text(
    0.25, 0.15,
    f"Slope for exponential supression = {slope:.3f}",
    transform=plt.gca().transAxes,
    fontsize=9,
    verticalalignment='top',
    bbox=dict(facecolor='white', alpha=0.8)
)
plt.title("Memory_Z Experiment, [7,4,3] Outer Code")

plt.show()




So we have shown how to find the logical error rate for a specific elevator code. For out final experiment, lets see what effect increasing the Z distance has on the X logical error rate.

In [ ]:

num_shots = 1000000
ps = np.logspace(-6,-5,5)
ds = [9,11,13,15]

err_rates=[]

for d in ds:
    d_err_rates = []
    for p in ps:
        circuit=noisy_circ(d,1,p)

        dem = circuit.detector_error_model()

        decoder, L = nvidia_stim_decoder(dem)

        mats = detector_error_model_to_check_matrices(dem)
        H = mats.check_matrix
        L = mats.observables_matrix

        sampler = circuit.compile_detector_sampler()
        shots, observables = sampler.sample(num_shots, separate_observables=True)

        predicted_observables=[]
        errors=decoder.decode_batch(shots) 

        for i in range(num_shots):
            result = errors[i].result
            data_prediction = np.array(result, dtype=np.uint8)
            predicted_observables.append((L @ data_prediction) % 2)
        num_mistakes = np.sum(np.any(predicted_observables != observables, axis=1))

        d_err_rates.append(num_mistakes/num_shots)

    err_rates.append(d_err_rates)

plt.plot(ps,err_rates[0], label="$d_Z$=9", marker = 'o')
plt.plot(ps,err_rates[1], label="$d_Z$=11", marker = 'o')
plt.plot(ps,err_rates[2], label="$d_Z$=13", marker = 'o')
plt.plot(ps,err_rates[3], label="$d_Z$=15", marker = 'o')
plt.loglog()
plt.legend()
plt.xlabel("Physical X Error Rate")
plt.ylabel("Logical X Error Rate")
plt.title("Increasing $d_Z$")

## Global Fit
Finally, we are going to fit a function for the X logical error rate $p_{X_L}$ of a [7,4,3] Hamming code ay physical error rate $p_X$ and Z distance $d_Z$. The fitting function we are going to use is:

$p_{X_L}(p_X, d_Z)=(d_Z)^a(bp_X)^{c}$

In [ ]:

def build_fit_dataframe(ps, ds, err_rates):
    ps = np.asarray(ps)
    n_p = len(ps)

    return pd.DataFrame({
        "logical_err": np.concatenate([np.asarray(e) for e in err_rates]),
        "d_z":         np.repeat(ds, n_p),
        "physical_err": np.tile(ps, len(ds)),
    })


def func(p, d, a, b, c):
    return ((b * p) ** c) * d ** a


def fit_logical_error(df, p0=None):
    p0 = p0 or dict(a=0.5, b=0.5, c=1.0)

    model = Model(func, independent_vars=["p", "d"])
    params = model.make_params(**p0)
    for name in ("a", "b", "c"):
        params[name].set(min=0)

    return model.fit(
        df["logical_err"].values,
        params,
        p=df["physical_err"].values,
        d=df["d_z"].values,
        nan_policy="omit",
        method="least_squares",
        max_nfev=int(1e6),
        weights=1 / df["logical_err"].values,
    )

df = build_fit_dataframe(ps, ds, err_rates)
result = fit_logical_error(df)
print(result.fit_report())

a = result.params["a"].value
b = result.params["b"].value
c = result.params["c"].value

In [ ]:
from matplotlib.lines import Line2D

for dv in ds:
    sub = df[df["d_z"] == dv].sort_values("physical_err")

    line, = plt.plot(sub["physical_err"], sub["logical_err"], "o", label=f"$d_Z$={dv} data")
    plt.plot(sub["physical_err"], func(sub["physical_err"], dv, a, b, c), "--", color=line.get_color())

plt.loglog()
plt.xlabel("Physical error rate")
plt.ylabel("Logical error rate")
plt.title(f"Fitting function for X logical error rate")

fit_proxy = Line2D([0], [0], color="black", linestyle="--", label="Global fit")
plt.legend(handles=plt.gca().get_legend_handles_labels()[0] + [fit_proxy])

So the final function we get for the X logical error rate is:

$p_{X_L}(p_X, d_Z)=(d_Z)^{3.33}(82.33p_X)^{1.87}$

# Additional Exercise
We have created a model for the logical X error rate of the [7,4,3] code. 

Now see if you can create a model for the larger [15,9,3] code. 

To help we have provided the noisy STIM circuit, just the same as above.

If you have any questions you can email me at P.A.Shanahan@sms.ed.ac.uk

Good luck!


In [ ]:
from Circuit_builder_advanced import noisy_circ_advanced

In [ ]:
circuit = noisy_circ_advanced(d=3, rs=1, p=1e-5).without_noise()


html_content = f"""
<div style="overflow-x: auto; overflow-y: hidden;">
    <div style="white-space: nowrap;width: {100}%;">{circuit.diagram('timeline-svg')}</div>
</div>
"""
display(HTML(html_content))